# Halcyon AUV — Training Notebook
**Runtime:** GPU → A100 (Runtime > Change runtime type > A100)

Run cells top to bottom. Each cell is labelled with what it does.

In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/rl_research/auv'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive mounted. Save root: {DRIVE_ROOT}')

In [ ]:
# CELL 2 — Install dependencies (~2 minutes, run once per session)
import subprocess, sys

packages = [
    'mujoco',
    'gymnasium[mujoco]',
    'stable-baselines3[extra]',
    'tensorboard',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import mujoco, gymnasium, stable_baselines3, torch
print(f'mujoco:            {mujoco.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {stable_baselines3.__version__}')
print(f'torch:             {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')

In [ ]:
# CELL 3 — Clone or update your GitHub repo
import os
from pathlib import Path

REPO_URL = 'https://github.com/Itzlimon22/Obhyash-complete-project'
REPO_DIR = Path('/content/rl_robotics')

if not REPO_DIR.exists():
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print(f'Cloned to {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')
    print(f'Pulled latest to {REPO_DIR}')

# Verify key files exist
required = [
    'envs/auv.xml',
    'envs/auv_env.py',
    'envs/auv_dr_wrapper.py',
    'scripts/train.py',
]
all_ok = True
for f in required:
    exists = (REPO_DIR / f).exists()
    print(f'  {"OK" if exists else "MISSING"}: {f}')
    if not exists:
        all_ok = False

if all_ok:
    print('All files present. Ready to train.')
else:
    print('ERROR: Missing files. Push them to GitHub and re-run this cell.')

In [ ]:
# CELL 4 — Sanity check (100 steps, ~10 seconds)
# Always run this before a long training job.
import sys
sys.path.insert(0, str(REPO_DIR / 'envs'))
sys.path.insert(0, str(REPO_DIR / 'scripts'))

from auv_env        import HalcyonAUVEnv
from auv_dr_wrapper import AUVDomainRandomWrapper

xml_path = str(REPO_DIR / 'envs' / 'auv.xml')
env = AUVDomainRandomWrapper(
    HalcyonAUVEnv(xml_path=xml_path),
    mode='curriculum', seed=0, verbose=False
)

obs, info = env.reset(seed=0)
print(f'obs shape:  {obs.shape}')
print(f'goal dist:  {info["goal_dist"]:.2f}m')

total_r = 0
for _ in range(100):
    obs, r, terminated, truncated, info = env.step(env.action_space.sample())
    total_r += r
    if terminated or truncated:
        obs, info = env.reset()

env.close()
print(f'100 steps done | cumulative reward: {total_r:.2f}')
print('Sanity check PASSED')

In [ ]:
# CELL 5 — Configure and launch training
#
# Change MODE and SEED for each paper experiment:
#   MODE='curriculum' SEED=0,1,2  → CDR (your contribution)
#   MODE='uniform'    SEED=0,1,2  → Uniform DR baseline
#   MODE='none'       SEED=0,1,2  → Naive SAC baseline
#
# Run one condition per Colab session to avoid timeouts.

import argparse
from train import train

# ── CONFIGURE HERE ───────────────────────────────────────
MODE  = 'curriculum'   # 'none' | 'uniform' | 'curriculum'
SEED  = 0              # 0, 1, or 2
STEPS = 1_000_000      # 1M for paper, 50_000 for debug
# ─────────────────────────────────────────────────────────

args = argparse.Namespace(
    mode     = MODE,
    seed     = SEED,
    steps    = STEPS,
    xml      = str(REPO_DIR / 'envs' / 'auv.xml'),
    run_name = f'{MODE}_seed{SEED}',
    save_dir = DRIVE_ROOT,
)

print(f'Starting: mode={MODE}  seed={SEED}  steps={STEPS:,}')
print(f'Saving to: {DRIVE_ROOT}/{MODE}/{MODE}_seed{SEED}')

save_dir = train(args)
print(f'Training complete: {save_dir}')

In [ ]:
# CELL 6 — TensorBoard (run during or after training)
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/rl_research/auv

In [ ]:
# CELL 7 — Quick eval: load best model, run 20 episodes
import numpy as np
from pathlib import Path
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Match these to whatever run you just trained
EVAL_MODE = MODE
EVAL_SEED = SEED
N_EVAL    = 20

run_dir    = Path(DRIVE_ROOT) / EVAL_MODE / f'{EVAL_MODE}_seed{EVAL_SEED}'
model_path = run_dir / 'best_model.zip'
stats_path = run_dir / 'vec_normalize.pkl'

if not model_path.exists():
    print(f'No model at {model_path} — run Cell 5 first.')
else:
    def make_eval_env():
        env = HalcyonAUVEnv(xml_path=str(REPO_DIR / 'envs' / 'auv.xml'))
        return AUVDomainRandomWrapper(env, mode=EVAL_MODE, seed=9999, verbose=False)

    vec = DummyVecEnv([make_eval_env])
    vec = VecNormalize.load(str(stats_path), vec)
    vec.training    = False
    vec.norm_reward = False

    model = SAC.load(str(model_path), env=vec, device='cuda')

    successes, rewards, dists = [], [], []
    obs = vec.reset()
    ep_r, ep_count = 0, 0

    while ep_count < N_EVAL:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, done, info = vec.step(action)
        ep_r += r[0]
        if done[0]:
            dist    = info[0].get('goal_dist', float('inf'))
            success = dist < 0.5
            successes.append(success)
            rewards.append(ep_r)
            dists.append(dist)
            ep_r = 0
            ep_count += 1

    vec.close()
    print(f'Mode: {EVAL_MODE}  Seed: {EVAL_SEED}')
    print(f'Success rate: {np.mean(successes):.1%}  ({sum(successes)}/{N_EVAL})')
    print(f'Mean reward:  {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}')
    print(f'Mean dist:    {np.mean(dists):.2f}m  +/- {np.std(dists):.2f}m')

In [ ]:
# CELL 8 — CDR curriculum state inspection
# Run mid-training or after to see how far the curriculum expanded.
import json
from pathlib import Path

run_dir   = Path(DRIVE_ROOT) / 'curriculum' / 'curriculum_seed0'
cdr_files = sorted(run_dir.glob('cdr_state_*.json'))

if not cdr_files:
    print('No CDR snapshots yet. Training must reach 50k steps first.')
else:
    print(f'{"Step":>10}  {"Level":>8}  {"Episodes":>10}  {"Successes":>10}')
    print('-' * 50)
    for f in cdr_files:
        state = json.load(open(f))
        step  = int(f.stem.split('_')[-1])
        print(
            f'{step:>10,}  '
            f'{state["curriculum_level"]:>8.3f}  '
            f'{state["n_episodes"]:>10,}  '
            f'{state["n_successes"]:>10,}'
        )

    final = json.load(open(cdr_files[-1]))
    print(f'\nFinal CDR ranges:')
    for k, (lo, hi) in final['cdr_ranges'].items():
        print(f'  {k:25s}: [{lo:.4f}, {hi:.4f}]')